# MÅL

Bruke reddits webapi til å:
- gjøre et søk på "elbiler"
- Hente alle artikler ("treff") fra det siste året
- Hente ut alle (nesten) kommentarer til innleggene
- putte det i pandas dataframe

evt:
- Gjøre sentimentanalyse med chatgpt api positiv/nøytral/negativ
- plotte utvikling (om noen)

In [1]:
import requests, json
import pandas as pd


In [2]:
with open("reddit_tokens.json", "r") as file:
    tokens = json.load(file)

access_token = tokens["access_token"]
refresh_token = tokens["refresh_token"]

client_id = "v2uZeXUHIszhF2K4hNOksQ"
client_secret = "2qamP2_KAEG7eNkXMwrJPhbb6jxzKw"

def refresh_tokens():
    global access_token, refresh_token
    refresh_url = "https://www.reddit.com/api/v1/access_token"
    payload = {"grant_type": "refresh_token", "refresh_token": refresh_token}
    headers = {"User-Agent": "python:undervisning_h25"}
    res = requests.post(refresh_url, auth=(client_id, client_secret), data=payload, headers=headers)
    res.raise_for_status()
    tokens = res.json()
    access_token = tokens["access_token"]
    refresh_token = tokens["refresh_token"]
    with open("reddit_tokens.json", "w") as file:
        json.dump(tokens,file)


def reddit_get(endpoint, params=None):
    base_url = "https://oauth.reddit.com"
    headers = {"Authorization": f"Bearer {access_token}",
              "User-Agent": "python:undervisning_h25"}
    res = requests.get(base_url+endpoint, headers=headers, params=params)
    res.raise_for_status()
    return res.json()

def reddit_search(q):
    params =  {
        "q": q,
        "sort": "top",
        "t": "year",
        "limit": 100
    }
    res_sider = []
    res = reddit_get("/search", params)
    res_sider.append(res)
    after = res["data"]["after"]
    
    while after:
        params = {
            "q": q,
            "sort": "top",
            "t": "year",
            "limit": 100,
            "after": after
        }
        res = reddit_get("/search", params)
        res_sider.append(res)
        after = res["data"]["after"]
    return res_sider



In [5]:
params = {
    "q": "elbil",
    "sort": "top",
    "t": "year",
    "limit": 100
}

treff = reddit_search("elbil")



In [6]:
dfs = [ pd.json_normalize(dat, record_path=["data","children"]) for dat in treff]
df = pd.concat(dfs)
kolonner = ["kind", "data.subreddit", "data.selftext", 
            "data.author_fullname", "data.title", 
            "data.name", "data.id", "data.created", "data.url"]
df_search = df[kolonner]
df_search = df_search.set_index(pd.PeriodIndex(pd.to_datetime(df_search["data.created"],unit="s", utc=True), freq="D"))
df_search = df_search.sort_index()
df_search

,kind,data.subreddit,data.selftext,data.author_fullname,data.title,data.name,data.id,data.created,data.url
data.created,,,,,,,,,
2024-12-30,t3,Denmark,,t2_31pqtxbc,Kommuner shopper kinesiske elbiler trods advar...,t3_1hpjdlv,1hpjdlv,1.735550e+09,https://nyheder.tv2.dk/business/2024-12-17-kom...
2024-12-31,t3,dkbiler,"Vi skal have ny bil til april, hvor vores nuvæ...",t2_11cqde,Import af elbil fra Tyskland,t3_1hqaed4,1hqaed4,1.735633e+09,https://www.reddit.com/r/dkbiler/comments/1hqa...
2025-01-01,t3,elbilsverige,"Hej,\n\nJag har den senaste månaden funderat l...",t2_14i5m8,Byta till elbil,t3_1hr5g73,1hr5g73,1.735746e+09,https://www.reddit.com/r/elbilsverige/comments...
2025-01-02,t3,Denmark,,t2_3t05pwo8,Elbiler overhalede salget af benzin- og diesel...,t3_1hrpn9s,1hrpn9s,1.735808e+09,https://www.dr.dk/nyheder/indland/elbiler-over...
2025-01-03,t3,electricvehicles,1. Tesla Model Y - 16858\t\n2. Tesla Model 3 ...,t2_1yk2g3cp,Top 20 car sales in Norway in 2024,t3_1hsp1oh,1hsp1oh,1.735918e+09,https://www.reddit.com/r/electricvehicles/comm...
...,...,...,...,...,...,...,...,...,...
2025-12-25,t3,dkbiler,https://fdm.dk/nyheder/nyt-om-trafik-og-biler/...,t2_jrmd6medd,Test: Varmepumpen har ingen effekt,t3_1pvaucz,1pvaucz,1.766659e+09,https://blog.bilbasen.dk/test-varmepumpen-har-...
2025-12-25,t3,algeria,"Salam guys,\nI hope you are all doing well and...",t2_1snoxpvfeh,Vaccination compaign in Algeria for children,t3_1pvhjqs,1pvhjqs,1.766682e+09,https://i.redd.it/mv71x9risd9g1.jpeg
2025-12-26,t3,norske,,t2_c5ud1v82,Når politikerne tror de er gode investorer med...,t3_1pw2w4g,1pw2w4g,1.766751e+09,https://i.redd.it/00jgvbk4ij9g1.png


In [8]:
treff[0]["data"]["children"][0]["data"]["selftext"]
#print("Antall treff", len(treff["data"]["children"]))
#for post in treff["data"]["children"]:
#    print(post["data"]["selftext"])
#    print("------------------------------\n\n")

'Etter å ha lest statistikken om hvor JÆVLIG mye Tesla vi nordmenn kjøper, kommer nok denne posten til å bli downvota til hælvete. Og jeg sier dette ekstra frustrert fordi foreldrene mine akkurat har kjøpt Tesla. Jeg vet de bare tenker på miljøet og økonomien. Men likevel, det føles bare så idiotisk. Salget i Europa har gått motsatt vei, men i Norge selges disse bilene som varmt hvetebrød... Vi i Norge liker å tro at vi er så moralsk overlegne når det kommer til amerikansk politikk. De aller fleste jeg snakker med synes DJT er en katastrofe. Egoisme, korrupsjon, ekkokamre, klimafornekting, splittelse. Likevel, når vi ser på hva som triller rundt på norske veier, er det Tesla overalt. Det er blitt nasjonalbilen? Og ja, jeg skjønner det er en elbil og at folk liker å tenke de gjør en miljøinnsats. Men på den andre siden, når du kjøper en Tesla, putter du penger rett i lomma på Elon Musk.\n\nOg Musk er grunnen til at DJT vant valget, og han flørter åpenlyst med høyreradikale miljøer, og s

In [9]:
sinnatype = """'Etter å ha lest statistikken om hvor JÆVLIG mye Tesla vi nordmenn kjøper, kommer nok denne posten til å bli downvota til hælvete. Og jeg sier dette ekstra frustrert fordi foreldrene mine akkurat har kjøpt Tesla. Jeg vet de bare tenker på miljøet og økonomien. Men likevel, det føles bare så idiotisk. Salget i Europa har gått motsatt vei, men i Norge selges disse bilene som varmt hvetebrød... Vi i Norge liker å tro at vi er så moralsk overlegne når det kommer til amerikansk politikk. De aller fleste jeg snakker med synes DJT er en katastrofe. Egoisme, korrupsjon, ekkokamre, klimafornekting, splittelse. Likevel, når vi ser på hva som triller rundt på norske veier, er det Tesla overalt. Det er blitt nasjonalbilen? Og ja, jeg skjønner det er en elbil og at folk liker å tenke de gjør en miljøinnsats. Men på den andre siden, når du kjøper en Tesla, putter du penger rett i lomma på Elon Musk.\n\nOg Musk er grunnen til at DJT vant valget, og han flørter åpenlyst med høyreradikale miljøer, og spiller en enorm rolle i å normalisere holdninger vi i utgangspunktet hater i Norge. \nVi snakker om at vi ikke vil støtte krefter som undergraver demokratiet, retten på ferie, overtidsbetaling eller pusher ekstreme agendaer i Europa, men så ender vi likevel med å finansiere en av de største mikrofonene disse kreftene har. For meg er det akkurat det samme som å si at vi er anti-røyk men likevel investere massivt i Philip Morris.  \n\n'"""
sinnatype

"'Etter å ha lest statistikken om hvor JÆVLIG mye Tesla vi nordmenn kjøper, kommer nok denne posten til å bli downvota til hælvete. Og jeg sier dette ekstra frustrert fordi foreldrene mine akkurat har kjøpt Tesla. Jeg vet de bare tenker på miljøet og økonomien. Men likevel, det føles bare så idiotisk. Salget i Europa har gått motsatt vei, men i Norge selges disse bilene som varmt hvetebrød... Vi i Norge liker å tro at vi er så moralsk overlegne når det kommer til amerikansk politikk. De aller fleste jeg snakker med synes DJT er en katastrofe. Egoisme, korrupsjon, ekkokamre, klimafornekting, splittelse. Likevel, når vi ser på hva som triller rundt på norske veier, er det Tesla overalt. Det er blitt nasjonalbilen? Og ja, jeg skjønner det er en elbil og at folk liker å tenke de gjør en miljøinnsats. Men på den andre siden, når du kjøper en Tesla, putter du penger rett i lomma på Elon Musk.\n\nOg Musk er grunnen til at DJT vant valget, og han flørter åpenlyst med høyreradikale miljøer, og 

In [12]:

testid = df_search.iloc[0,-3]
url = df_search.iloc[0,-1]

comments_url = "/comments/article"
params = {"article": testid}

kommentar = reddit_get(comments_url, params)

with open("kommentartest.json", "w") as file:
    json.dump(kommentar,file)



In [13]:
kommentarer = kommentar[1]["data"]["children"].copy()
out = []
while len(kommentarer) > 0:
    kom = kommentarer.pop()
    if kom["kind"] == "Listing":
        kommentarer.extend(kom["data"]["children"])
    else:
        out.append(kom["data"]["body"])
        if isinstance(kom["data"]["replies"], dict):
            kommentarer.append(kom["data"]["replies"])



In [10]:
def get_comments(artikkel_id):
    comments_url = "/comments/article"
    params = {"article": artikkel_id}
    data = reddit_get(comments_url, params)
    kommentarer = data[1]["data"]["children"].copy()
    out = []
    while len(kommentarer) > 0:
        kom = kommentarer.pop()
        if kom["kind"] == "Listing":
            kommentarer.extend(kom["data"]["children"])
        elif kom["kind"] == "t1":
            out.append(kom["data"]["body"])
            if isinstance(kom["data"]["replies"], dict):
                kommentarer.append(kom["data"]["replies"])
    return out

testid = "1nqsvr8"
kommentarer = get_comments(testid)


In [11]:
%%time
df_search["kommentarer"] = df_search["data.id"].map(get_comments)


CPU times: user 3.28 s, sys: 232 ms, total: 3.52 s
Wall time: 2min 41s


In [12]:
df_search.set_index("data.id")
n_comments = 243

n_comments += df_search["kommentarer"].map(len).sum()
print("antall innlegg + kommentarer = ", n_comments)

antall innlegg + kommentarer =  15532


In [13]:
df_search.index.name = "dato"
df = df_search.copy()

In [14]:
df = df.explode("kommentarer")
df

,kind,data.subreddit,data.selftext,data.author_fullname,data.title,data.name,data.id,data.created,data.url,kommentarer
dato,,,,,,,,,,
2024-12-30,t3,Denmark,,t2_31pqtxbc,Kommuner shopper kinesiske elbiler trods advar...,t3_1hpjdlv,1hpjdlv,1.735550e+09,https://nyheder.tv2.dk/business/2024-12-17-kom...,Slettemette skal ikke bestemme hvilke biler ko...
2024-12-30,t3,Denmark,,t2_31pqtxbc,Kommuner shopper kinesiske elbiler trods advar...,t3_1hpjdlv,1hpjdlv,1.735550e+09,https://nyheder.tv2.dk/business/2024-12-17-kom...,Du skal lige starte med at få den godkendt først.
2024-12-30,t3,Denmark,,t2_31pqtxbc,Kommuner shopper kinesiske elbiler trods advar...,t3_1hpjdlv,1hpjdlv,1.735550e+09,https://nyheder.tv2.dk/business/2024-12-17-kom...,Desværre har EUmafiaen set sig rigtig sure på ...
2024-12-30,t3,Denmark,,t2_31pqtxbc,Kommuner shopper kinesiske elbiler trods advar...,t3_1hpjdlv,1hpjdlv,1.735550e+09,https://nyheder.tv2.dk/business/2024-12-17-kom...,Ja det er helt sikkert derfor. Cybertruck er j...
2024-12-30,t3,Denmark,,t2_31pqtxbc,Kommuner shopper kinesiske elbiler trods advar...,t3_1hpjdlv,1hpjdlv,1.735550e+09,https://nyheder.tv2.dk/business/2024-12-17-kom...,Løgn https://cleantechnica.com/2024/10/15/tesl...
...,...,...,...,...,...,...,...,...,...,...
2025-12-28,t3,Denmark,Jeg er for nylig blevet den lykkelige ejer af ...,t2_1815xp9d7l,Hvorfor bliver folk ved med at tale om deres b...,t3_1pxmaae,1pxmaae,1.766910e+09,https://www.reddit.com/r/Denmark/comments/1pxm...,Tværtimod. Du vågner med lukket dør. Luft til ...
2025-12-28,t3,Denmark,Jeg er for nylig blevet den lykkelige ejer af ...,t2_1815xp9d7l,Hvorfor bliver folk ved med at tale om deres b...,t3_1pxmaae,1pxmaae,1.766910e+09,https://www.reddit.com/r/Denmark/comments/1pxm...,Nej men du kan sagtens sætte varmepumpen til f...
2025-12-28,t3,Denmark,Jeg er for nylig blevet den lykkelige ejer af ...,t2_1815xp9d7l,Hvorfor bliver folk ved med at tale om deres b...,t3_1pxmaae,1pxmaae,1.766910e+09,https://www.reddit.com/r/Denmark/comments/1pxm...,Når jeg hører strøm til under 1 kr/kwh så må d...


In [15]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from openai import OpenAI
import os

# 1. The inner-most model
class User(BaseModel):
    name: str = Field(description="The full name of the person")
    role: str = Field(description="Their job title or department")

# 2. The middle-layer model
class Task(BaseModel):
    title: str
    priority: Literal["high", "medium", "low"]
    assignee: User  # Nesting the User model here

# 3. The root model
class Project(BaseModel):
    project_name: str
    deadline: str
    tasks: List[Task] # A list of nested Task objects
    budget_approved: bool

Project.model_json_schema()

{'$defs': {'Task': {'properties': {'title': {'title': 'Title',
     'type': 'string'},
    'priority': {'enum': ['high', 'medium', 'low'],
     'title': 'Priority',
     'type': 'string'},
    'assignee': {'$ref': '#/$defs/User'}},
   'required': ['title', 'priority', 'assignee'],
   'title': 'Task',
   'type': 'object'},
  'User': {'properties': {'name': {'description': 'The full name of the person',
     'title': 'Name',
     'type': 'string'},
    'role': {'description': 'Their job title or department',
     'title': 'Role',
     'type': 'string'}},
   'required': ['name', 'role'],
   'title': 'User',
   'type': 'object'}},
 'properties': {'project_name': {'title': 'Project Name', 'type': 'string'},
  'deadline': {'title': 'Deadline', 'type': 'string'},
  'tasks': {'items': {'$ref': '#/$defs/Task'},
   'title': 'Tasks',
   'type': 'array'},
  'budget_approved': {'title': 'Budget Approved', 'type': 'boolean'}},
 'required': ['project_name', 'deadline', 'tasks', 'budget_approved'],
 '

In [43]:
from pydantic import BaseModel, Field
from openai import OpenAI
import os, time


class analyse_kommentar(BaseModel):
    nummer: int = Field("kommentarnummer")
    analyse: int = Field("Sentimentanalyse av kommentar #nummer# (0 nøytral, -1 negativ, +1 positiv")

class sentimentanalyse(BaseModel):
    post: int = Field("sentimentanalyse av hovedartikkel: 0 nøytral/ikke-relevant, -1 negativ, +1 positiv")
    kommentar: list[analyse_kommentar]


client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def analyser(komm):
    time.sleep(.5)
    
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Du er ekspert på sentimentanalyse av holdninger rundt elbiler og det grønne skiftet og ser på lister med innlegg og kommentarer fra reddit på norsk, dansk og svensk og vurderer om de er positiv, negativ eller nøytral til elbiler"},
            {"role": "user", "content": f"Under er en redditpost sammen med kommentarfeltet i en liste av strenger - du skal vurdere om kommentarene eller person som skriver de er positiv, negativ eller nøytral/ikke-relevant til elbiler. 1 for positiv, -1 for negativ og 0 for nøytral, eller om artikkel/kommentar ikke er relevant\n{komm}"}
        ],
        response_format=sentimentanalyse
    )
    return completion


In [42]:

log = []

def analyser_post(df_post):
    
    dat = {"post": df_post["data.selftext"].iloc[0]}
    dat.update({str(i): kommentar for i,kommentar in enumerate(list(df_post["kommentarer"]))})

    resp = analyser(dat)
    data_resp = resp.choices[0].message.parsed.model_dump()
    log.append(data_resp)
    if data_resp["kommentar"]:
        ny_df = pd.json_normalize(data_resp, "kommentar", meta="post").drop(columns="nummer")
    else:
        ny_df = pd.Series({"post": data_resp["post"], "analyse": np.nan} ).to_frame().T
    return ny_df


test = df.query("`data.title`.str.contains('Låne i frivæ')")
#test_dat = list(test["data.selftext"].unique())
#test_dat.extend(list(test["kommentarer"]))

#test_dat = {"post": test["data.selftext"].iloc[0]}
#kommentarer = {f"{i}": kommentar for i,kommentar in enumerate(list(test["kommentarer"]))}

#test_dat.update(kommentarer)

#resp = analyser(test_dat)

In [40]:
data = resp.choices[0].message.parsed.model_dump()

NameError: name 'resp' is not defined

In [39]:
d = pd.json_normalize(data, "kommentar", meta="post").drop(columns="nummer")
tmp = pd.concat([d.set_index(test.index), test], axis=1)




NameError: name 'data' is not defined

In [44]:
df_test = df.query("`data.selftext`.str.contains('Jeg tog turen')")
test_res = df_test.groupby(by="data.id").apply(analyser_post, include_groups=False)


In [46]:
%%time
big = df.groupby(by="data.id").apply(analyser_post, include_groups=False)
big.to_csv("sentimentanalyse.csv")

CPU times: user 4.97 s, sys: 97.3 ms, total: 5.07 s
Wall time: 45min 40s


[{'post': 0,
  'kommentar': [{'nummer': 0, 'analyse': 0},
   {'nummer': 1, 'analyse': 0},
   {'nummer': 2, 'analyse': 0},
   {'nummer': 3, 'analyse': -1},
   {'nummer': 4, 'analyse': 0},
   {'nummer': 5, 'analyse': 0},
   {'nummer': 6, 'analyse': 0},
   {'nummer': 7, 'analyse': 0},
   {'nummer': 8, 'analyse': 1},
   {'nummer': 9, 'analyse': 1},
   {'nummer': 10, 'analyse': -1},
   {'nummer': 11, 'analyse': 0},
   {'nummer': 12, 'analyse': 0},
   {'nummer': 13, 'analyse': 0},
   {'nummer': 14, 'analyse': -1},
   {'nummer': 15, 'analyse': 1},
   {'nummer': 16, 'analyse': 1},
   {'nummer': 17, 'analyse': 0},
   {'nummer': 18, 'analyse': 0},
   {'nummer': 19, 'analyse': 0},
   {'nummer': 20, 'analyse': 0},
   {'nummer': 21, 'analyse': 0},
   {'nummer': 22, 'analyse': 1},
   {'nummer': 23, 'analyse': 0},
   {'nummer': 24, 'analyse': 0},
   {'nummer': 25, 'analyse': 0},
   {'nummer': 26, 'analyse': 0},
   {'nummer': 27, 'analyse': 0},
   {'nummer': 28, 'analyse': 0},
   {'nummer': 29, 'analy

In [37]:
edgecase = {'post': 0, 'kommentar': []}
import numpy as np
pd.Series({"post": int(edgecase["post"]), "analyse": np.nan} ).to_frame().T

,post,analyse
0,0.0,NaN


In [25]:
regular =  {'post': 1,
  'kommentar': [{'nummer': 0, 'analyse': -1},
   {'nummer': 1, 'analyse': 0},
   {'nummer': 2, 'analyse': 0},
   {'nummer': 3, 'analyse': 0},
   {'nummer': 4, 'analyse': 0},
   {'nummer': 5, 'analyse': 1},
   {'nummer': 6, 'analyse': 1},
   {'nummer': 7, 'analyse': 1},
   {'nummer': 8, 'analyse': 1},
   {'nummer': 9, 'analyse': 1},
   {'nummer': 10, 'analyse': 1},
   {'nummer': 11, 'analyse': 1},
   {'nummer': 12, 'analyse': 1},
   {'nummer': 13, 'analyse': 1},
   {'nummer': 14, 'analyse': 1},
   {'nummer': 15, 'analyse': 1},
   {'nummer': 16, 'analyse': 1},
   {'nummer': 17, 'analyse': 1},
   {'nummer': 18, 'analyse': 1},
   {'nummer': 19, 'analyse': 1},
   {'nummer': 20, 'analyse': 1}]}
pd.json_normalize(regular, "kommentar", meta="post").drop(columns="nummer")

,analyse,post
0,-1,1
1,0,1
2,0,1
3,0,1
4,0,1
5,1,1
6,1,1
7,1,1
8,1,1
9,1,1


In [45]:
test_res

analyse post
data.id                 
1phc5l0 0         1    0
        1        -1    0
        2         1    0
        3         1    0
        4         1    0
        5         1    0
        6        -1    0
        7        -1    0
        8         1    0
        9        -1    0
        10        1    0
        11        1    0
        12        1    0
        13        1    0
        14        1    0
        15        0    0
        16        0    0
        17        1    0
        18        1    0
        19       -1    0
        20        1    0
        21        0    0
        22        0    0
        23        0    0
        24       -1    0
        25        1    0
        26        0    0
        27       -1    0
        28        1    0
        29        1    0
        30        1    0
        31        1    0
        32        0    0
        33        1    0
        34        0    0
        35        1    0
        36        1    0
        37        1    0
        38       -1    0
        39        1    0
        40        1    0
        41        0    0
        42        0    0
        43        1    0
        44        1    0
        45       -1    0
        46        1    0
        47        1    0
        48       -1    0
        49        0    0
        50        0    0
        51        0    0
        52        1    0
        53        0    0
        54        0    0
        55        0    0
        56        0    0
        57        1    0
        58        1    0
        59        0    0